# Deep Tabular Baseline Models: SAINT and FT-Transformer

* **Objective:** Establish baseline predictive performance using state-of-the-art deep learning architectures for tabular data.
* **Architectures Evaluated:**
  * **SAINT:** Self-Attention and Intersample Attention Transformer.
  * **FT-Transformer:** Feature Tokenizer + Transformer.
* **Key Frameworks:** * `pytorch-widedeep`: Core library for deep tabular modeling.
  * `optuna`: Framework for automated hyperparameter optimization.

In [ ]:
# ==========================================
# [1] Required package installation & Setup
# ==========================================
!pip install pytorch-tabnet pytorch-widedeep optuna -q

import pandas as pd
import numpy as np
import torch
import optuna
import io
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# WideDeep modules for Deep Tabular learning
from pytorch_widedeep.preprocessing import TabPreprocessor
from pytorch_widedeep.models import WideDeep, SAINT, FTTransformer
from pytorch_widedeep import Trainer
from pytorch_widedeep.callbacks import EarlyStopping

# ==========================================
# [1-1] Data loading and preprocessing
# ==========================================
def load_data():
    """
    Load the harmonized dataset used for deep tabular baseline experiments.
    Prompts manual upload if running in Google Colab and file is missing.
    """
    try:
        from google.colab import files
        import os
        if not os.path.exists('Dataset.csv'):
            print("=== [Google Colab] Please upload Dataset.csv ===")
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
        else:
            df = pd.read_csv('Dataset.csv')
    except ImportError:
        df = pd.read_csv('Dataset.csv')
    return df

df = load_data()

# Drop redundant identifier if present
if 'Country Name' in df.columns:
    df = df.drop('Country Name', axis=1)

cat_cols = ['Country Code', 'Continent']
target_col = 'Maternal Mortality Ratio'

# Missing-value treatment
# - Categorical variables: Imputed with "Unknown"
# - Numerical variables: Imputed with the median value
for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

num_cols = [c for c in df.columns if c not in cat_cols + ['Year', target_col]]
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# ==========================================
# [1-2] Temporal split design
# ==========================================
# Phase 1: Model selection (Hyperparameter Tuning)
# Train: 2011-2014 / Validation: 2015
train_opt = df[(df['Year'] >= 2011) & (df['Year'] <= 2014)]
val_opt   = df[df['Year'] == 2015]

# Phase 2: Retraining after model selection
# Train: 2011-2015
train_retrain = df[(df['Year'] >= 2011) & (df['Year'] <= 2015)]

# Phase 3: Held-out test evaluation
# Test: 2016
test = df[df['Year'] == 2016]

feature_names = [c for c in df.columns if c not in ['Year', target_col]]

## 1. WideDeep Preprocessing & Bootstrap Evaluation

* **TabPreprocessor Configuration:**
  * Fits the preprocessing pipeline (scaling, categorical embeddings) strictly on the optimization-training period to avoid data leakage.
* **Bootstrap Confidence Intervals:**
  * A robust statistical evaluation method calculating 95% confidence intervals for MAE, RMSE, and R2 using 1,000 resampling iterations.

In [ ]:
# ==========================================
# [2] WideDeep preprocessing (TabPreprocessor)
# ==========================================
# Fit the preprocessing pipeline only on the optimization-training period
tab_preprocessor = TabPreprocessor(
    cat_embed_cols=cat_cols,
    continuous_cols=num_cols,
    scale=True
)
tab_preprocessor.fit(train_opt)

# Transform the full dataset
X_tab_all = tab_preprocessor.transform(df)
y_all = df[target_col].values

# Extract row indices for each temporal split
idx_opt_train = train_opt.index
idx_opt_val   = val_opt.index
idx_retrain   = train_retrain.index
idx_test      = test.index

# Input parameters required by SAINT / FT-Transformer models
input_params = {
    "column_idx": tab_preprocessor.column_idx,
    "cat_embed_input": tab_preprocessor.cat_embed_input,
    "continuous_cols": num_cols,
}

# ==========================================
# Bootstrap Confidence Interval Function
# ==========================================
def calculate_bootstrap_ci(y_true, y_pred, n_bootstraps=1000, ci=95, random_state=42):
    """
    Calculate the mean performance and confidence intervals using bootstrapping.

    Parameters:
    - y_true: Ground truth target values (1D array)
    - y_pred: Model predictions (1D array)
    - n_bootstraps: Number of resampling iterations (default 1000)
    - ci: Confidence interval percentage (default 95%)
    - random_state: Seed for reproducibility

    Returns:
    - Dictionary mapping metric names to 'Mean (Lower_Bound - Upper_Bound)' formatted strings
    """
    # Ensure consistent array shapes
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    n_samples = len(y_true)
    rng = np.random.RandomState(random_state)

    # Lists to store metric values per iteration
    boot_mae, boot_rmse, boot_r2 = [], [], []

    for _ in range(n_bootstraps):
        # 1. Random sampling with replacement
        indices = rng.randint(0, n_samples, n_samples)

        # 2. Create virtual test set
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # 3. Compute and store metrics
        boot_mae.append(mean_absolute_error(y_true_boot, y_pred_boot))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        boot_r2.append(r2_score(y_true_boot, y_pred_boot))

    # Calculate percentiles for lower and upper bounds
    lower_p = (100 - ci) / 2.0
    upper_p = 100 - lower_p

    results = {}
    metrics = {'MAE': boot_mae, 'RMSE': boot_rmse, 'R2': boot_r2}

    for name, values in metrics.items():
        mean_val = np.mean(values)
        lower_val = np.percentile(values, lower_p)
        upper_val = np.percentile(values, upper_p)

        results[name] = f"{mean_val:.4f} ({lower_val:.4f} - {upper_val:.4f})"

    return results

## 2. Unified Experiment Pipeline

* **Pipeline Workflow:**
  * **Step 1 (Optuna):** Searches for optimal hyperparameters using the validation set (2015).
  * **Step 2 (Retraining):** Trains a final model on the full pretest dataset (2011-2015) using the best-found parameters.
  * **Step 3 (Evaluation):** Tests on unseen 2016 data to measure final metrics and Bootstrap CIs.
  * **Step 4 (XAI):** Calculates permutation-based feature importance to interpret model behavior.

In [ ]:
# ==========================================
# [3] Unified experiment pipeline
# ==========================================
def run_experiment(model_name):
    """
    Execute a complete deep tabular baseline experiment.
    """
    print(f"\n{'='*10} Processing {model_name} {'='*10}")

    # --- Step 1: Hyperparameter optimization ---
    def objective(trial):
        # Define hyperparameter search spaces based on the architecture
        if model_name == 'SAINT':
            params = {
                'n_blocks': trial.suggest_int('n_blocks', 1, 3),
                'n_heads': trial.suggest_categorical('n_heads', [2, 4, 8]),
                'attn_dropout': trial.suggest_float('attn_dropout', 0.0, 0.3),
                'ff_dropout': trial.suggest_float('ff_dropout', 0.0, 0.3),
                'input_dim': 64
            }
            tab_model = SAINT(**input_params, **params)

        else:  # FT-Transformer
            params = {
                'n_blocks': trial.suggest_int('n_blocks', 2, 4),
                'n_heads': trial.suggest_categorical('n_heads', [2, 4, 8]),
                'attn_dropout': trial.suggest_float('attn_dropout', 0.0, 0.3),
                'ff_dropout': trial.suggest_float('ff_dropout', 0.0, 0.3),
                'input_dim': 32
            }
            tab_model = FTTransformer(**input_params, **params)

        # Wrap the tabular model within the WideDeep interface
        model = WideDeep(deeptabular=tab_model)

        # Train on 2011-2014, Validate on 2015
        trainer = Trainer(model, objective="regression", verbose=0)
        trainer.fit(
            X_train={"X_tab": X_tab_all[idx_opt_train], "target": y_all[idx_opt_train]},
            X_val={"X_tab": X_tab_all[idx_opt_val], "target": y_all[idx_opt_val]},
            n_epochs=15,
            batch_size=128,
            callbacks=[EarlyStopping(patience=5, monitor="val_loss")]
        )
        return trainer.history['val_loss'][-1]

    print(">> [Step 1] Optimizing hyperparameters...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=10) # 10 trials for demonstration
    best_params = study.best_trial.params
    print(f"   Best parameters: {best_params}")

    # --- Step 2: Retraining on 2011-2015 ---
    print(">> [Step 2] Retraining on the full pretest period (2011-2015)...")

    if model_name == 'SAINT':
        final_tab_model = SAINT(**input_params, input_dim=64, **best_params)
    else:
        final_tab_model = FTTransformer(**input_params, input_dim=32, **best_params)

    final_model = WideDeep(deeptabular=final_tab_model)

    # Train on the full pretest period without early stopping/validation split
    trainer = Trainer(final_model, objective="regression", verbose=0)
    trainer.fit(
        X_train={"X_tab": X_tab_all[idx_retrain], "target": y_all[idx_retrain]},
        n_epochs=30,
        batch_size=128
    )

    # --- Step 3: Final evaluation on the held-out year 2016 ---
    print(">> [Step 3] Evaluating on the held-out test year (2016)...")
    preds = trainer.predict(X_tab=X_tab_all[idx_test])
    y_true = y_all[idx_test]

    mae = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    r2 = r2_score(y_true, preds)

    # Calculate 95% Bootstrap CI
    boot_ci = calculate_bootstrap_ci(y_true, preds)
    print(f"   [Bootstrap 95% CI] MAE: {boot_ci['MAE']}, RMSE: {boot_ci['RMSE']}, R2: {boot_ci['R2']}")

    # --- Step 4: Permutation-based feature importance ---
    print(">> [Step 4] Calculating permutation importance...")
    base_score = r2_score(y_true, preds)
    importances = {}

    X_test_curr = X_tab_all[idx_test]
    for i, col in enumerate(feature_names):
        X_p = X_test_curr.copy()
        # Randomly shuffle a single feature column
        np.random.shuffle(X_p[:, i])
        preds_p = trainer.predict(X_tab=X_p)
        # Importance is defined as the drop in R2 score
        importances[col] = base_score - r2_score(y_true, preds_p)

    return {
        'Metrics': {'MAE': mae, 'RMSE': rmse, 'R2': r2},
        'Bootstrap_CI': boot_ci,
        'Importance': pd.Series(importances).sort_values(ascending=False),
        'Preds': preds
    }

## 3. Execution, Results, and Visualizations

* **Summary Metrics:**
  * Outputs side-by-side comparisons of standard metrics and 95% Bootstrap Confidence Intervals for both SAINT and FT-Transformer.
* **Graphical Comparisons:**
  * Dual-axis bar charts to visualize predictive error (MAE, RMSE) vs. explanatory power (R-Squared).
  * Feature importance bar plots to interpret which features strongly influence each model's predictions.

In [ ]:
# ==========================================
# [4] Run experiments and summarize outputs
# ==========================================
results_saint = run_experiment('SAINT')
results_ft = run_experiment('FT-Transformer')

# --- Summary tables ---
metrics_df = pd.DataFrame({
    'SAINT': results_saint['Metrics'],
    'FT-Transformer': results_ft['Metrics']
}).T[['MAE', 'RMSE', 'R2']]

bootstrap_df = pd.DataFrame({
    'SAINT': results_saint['Bootstrap_CI'],
    'FT-Transformer': results_ft['Bootstrap_CI']
}).T[['MAE', 'RMSE', 'R2']]

# Print structured final results
print("\n" + "="*50)
print(" FINAL RESULTS (Held-out Test Year: 2016) ")
print("="*50)
print(metrics_df)

print("\n" + "="*60)
print(" FINAL RESULTS (1000 Bootstraps 95% Confidence Interval) ")
print("="*60)
print(bootstrap_df)

# ==========================================
# [5] Visualization 1: Performance comparison
# ==========================================
fig, ax1 = plt.subplots(figsize=(10, 6))

# Error metrics (Left Axis)
metrics_df[['MAE', 'RMSE']].plot(
    kind='bar', ax=ax1, width=0.4, position=1, color=['#00cec9', '#fab1a0']
)
ax1.set_ylabel("Error (MAE / RMSE)", fontsize=12)
ax1.set_title("Performance Comparison: SAINT vs FT-Transformer", fontsize=14, fontweight='bold')

# R2 Score (Right Axis)
ax2 = ax1.twinx()
metrics_df['R2'].plot(
    kind='bar', ax=ax2, width=0.2, position=0, color='#6c5ce7', label='R2 Score'
)
ax2.set_ylabel("R2 Score", color='#6c5ce7', fontsize=12)
ax2.set_ylim(0, 1.1)

# Ensure legends don't overlap
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

# ==========================================
# [6] Visualization 2: Permutation importance
# ==========================================
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

# SAINT feature importance
results_saint['Importance'].nlargest(10).sort_values().plot(
    kind='barh', ax=axs[0], color='#00cec9', edgecolor='0.25'
)
axs[0].set_title("SAINT Feature Importance (Permutation)", fontweight='bold')
axs[0].set_xlabel("R2 Degradation")

# FT-Transformer feature importance
results_ft['Importance'].nlargest(10).sort_values().plot(
    kind='barh', ax=axs[1], color='#fab1a0', edgecolor='0.25'
)
axs[1].set_title("FT-Transformer Feature Importance (Permutation)", fontweight='bold')
axs[1].set_xlabel("R2 Degradation")

# Clean aesthetics
for ax in axs:
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()